# 15 · Model Export

**Purpose**: Artifact integrity check, score the full dataset, and
document the production integration plan.

| I/O | Path |
|-----|------|
| Input  | `data/processed/features.parquet` |
| Input  | `model/lgbm_atd_model.pkl` |
| Input  | `model/model_metadata.json` |
| Output | `data/processed/scored_dataset.parquet` |

---
## 1 · Setup

In [ ]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

FEAT_PATH    = Path('../data/processed/features.parquet')
MODEL_PATH   = Path('../model/lgbm_atd_model.pkl')
META_PATH    = Path('../model/model_metadata.json')
SCORED_PATH  = Path('../data/processed/scored_dataset.parquet')

# Columns required in scored output (feeds Streamlit dashboard)
OUTPUT_COLS = [
    'workflow_uuid',
    'driver_uuid',
    'delivery_trip_uuid',
    'restaurant_offered_timestamp_utc',
    'territory',
    'courier_flow',
    'geo_archetype',
    'merchant_surface',
    'ATD',
    'ATD_predicted',
    'hour_local',
    'is_peak_hour',
]

---
## 2 · Load Model & Metadata

In [ ]:
with open(META_PATH, 'r', encoding='utf-8') as fh:
    meta = json.load(fh)

model = joblib.load(MODEL_PATH)

FEATURE_COLS = meta['feature_cols']
CAT_FEATURES = meta['cat_features']
TARGET       = meta['target']

print(f'Model loaded from  : {MODEL_PATH}')
print(f'Best iteration     : {meta["best_iteration"]}')
print(f'Val MAE            : {meta["val_mae"]:.3f} min')
print(f'Feature count      : {len(FEATURE_COLS)}')

---
## 3 · Artifact Integrity Check (5-row Sanity Prediction)

In [ ]:
df_full = pd.read_parquet(FEAT_PATH)
df_full['restaurant_offered_timestamp_utc'] = pd.to_datetime(
    df_full['restaurant_offered_timestamp_utc']
)

for col in CAT_FEATURES:
    if col in df_full.columns:
        df_full[col] = df_full[col].astype('category')

# 5-row sanity prediction
sanity_rows = df_full.head(5)
X_sanity = sanity_rows[
    [c for c in FEATURE_COLS if c in sanity_rows.columns]
]
pred_sanity = np.clip(
    model.predict(X_sanity, num_iteration=meta['best_iteration']),
    a_min=0, a_max=None,
)

sanity_df = pd.DataFrame({
    'ATD_actual':    sanity_rows[TARGET].values,
    'ATD_predicted': pred_sanity.round(2),
})
print('Sanity predictions (first 5 rows):')
print(sanity_df.to_string(index=False))
assert all(pred_sanity >= 0), 'Negative predictions — artifact corrupt!'
print('\nArtifact integrity check passed.')

---
## 4 · Score Full Dataset

In [ ]:
X_full = df_full[
    [c for c in FEATURE_COLS if c in df_full.columns]
]

print(f'Scoring {len(df_full):,} rows...')
df_full['ATD_predicted'] = np.clip(
    model.predict(X_full, num_iteration=meta['best_iteration']),
    a_min=0, a_max=None,
)
print('Scoring complete.')

# Quick distribution check
print(
    f'ATD_predicted — '
    f'mean={df_full["ATD_predicted"].mean():.1f}  '
    f'median={df_full["ATD_predicted"].median():.1f}  '
    f'p90={df_full["ATD_predicted"].quantile(0.9):.1f}'
)

---
## 5 · Save `scored_dataset.parquet`

In [ ]:
# Keep only required output columns that exist in the dataframe
save_cols = [c for c in OUTPUT_COLS if c in df_full.columns]
missing = [c for c in OUTPUT_COLS if c not in df_full.columns]
if missing:
    print(f'Warning: missing columns skipped: {missing}')

df_scored = df_full[save_cols].copy()

# Convert categoricals to string for broad parquet compatibility
for col in CAT_FEATURES:
    if col in df_scored.columns:
        df_scored[col] = df_scored[col].astype(str)

df_scored.to_parquet(SCORED_PATH, index=False, engine='pyarrow')
size_mb = SCORED_PATH.stat().st_size / 1024 ** 2
print(f'scored_dataset.parquet → {SCORED_PATH}')
print(f'  Shape  : {df_scored.shape}')
print(f'  Size   : {size_mb:.1f} MB')
print(f'  Schema :')
for col in df_scored.columns:
    print(f'    {col:<45} {df_scored[col].dtype}')

---
## 6 · Final Summary Card

In [ ]:
sla_accuracy = float(
    np.mean(df_full['ATD_predicted'] <= meta['sla_threshold_min']) * 100
)

print('═' * 62)
print('  MODEL EXPORT SUMMARY')
print('═' * 62)
print(f'  Model type          : LightGBM (regression_l1)')
print(f'  Best iteration      : {meta["best_iteration"]}')
print(f'  Feature count       : {len(FEATURE_COLS)}')
print(f'    Numeric           : {len(meta["numeric_features"])}')
print(f'    Categorical       : {len(meta["cat_features"])}')
print()
print(f'  Val   MAE           : {meta["val_mae"]:.3f} min')
print(f'  Val   RMSE          : {meta["val_rmse"]:.3f} min')
print(f'  Val   MAPE          : {meta["val_mape"]:.2f}%')
print(f'  Val   R²            : {meta["val_r2"]:.4f}')
print()
print(f'  SLA threshold       : {meta["sla_threshold_min"]} min')
print(f'  SLA accuracy (full) : {sla_accuracy:.1f}%')
print()
print(f'  Output file         : scored_dataset.parquet')
print(f'  Output rows         : {len(df_scored):,}')
print(f'  Output columns      : {list(df_scored.columns)}')
print('═' * 62)

---
## 7 · Production ETL Integration Plan

### Overview

The `scored_dataset.parquet` produced by this notebook is the **static
scored file** consumed by the Streamlit dashboard.  In production, model
inference runs as a weekly scheduled step inside the Azure Data Factory
(ADF) pipeline documented in `documents/adf_pipeline.md`.

### Where inference runs in the ADF pipeline

```
ADF Pipeline: uber_eats_atd_weekly
  ├── Activity 1 · IngestRawCSV
  │     Copy Activity: Blob → ADLS Gen2 raw zone
  ├── Activity 2 · DataCleaning
  │     Databricks notebook: equivalent of nb 10
  ├── Activity 3 · FeatureEngineering
  │     Databricks notebook: equivalent of nb 11 + 12
  │     Writes features_latest.parquet to processed zone
  ├── Activity 4 · ModelInference   ← inference step
  │     Databricks notebook:
  │       - Loads lgbm_atd_model.pkl from model registry
  │       - Calls model.predict(features_latest)
  │       - Appends ATD_predicted column
  │       - Writes scored_dataset.parquet to processed zone
  └── Activity 5 · PublishToAATables
        Copy Activity: processed zone → AA_tables output container
```

### How predictions land in AA_tables

The `PublishToAATables` copy activity writes
`scored_dataset.parquet` to the `AA_tables/` output container under
a date-partitioned path:

```
AA_tables/
  scored_dataset/
    year=2025/week=14/scored_dataset.parquet
```

The dashboard always reads the **latest partition** (or the static
`data/processed/scored_dataset.parquet` when running locally).

### How `loader.py` picks up the file

`app/tools/dashboard/data/loader.py` calls `load_data(path)` where
`path` is resolved at runtime:

```python
# In app.py (or config.yaml)
SCORED_PATH = os.environ.get(
    'SCORED_PARQUET_PATH',
    'data/processed/scored_dataset.parquet',
)
df = load_data(SCORED_PATH)
```

When deployed on Azure, `SCORED_PARQUET_PATH` is set to the
Blob Storage URL (via `BLOB_SAS_URL`); locally it falls back to
the parquet written by this notebook.

The prediction tab in the dashboard reads `ATD_predicted` directly
from the scored file — no inference happens at dashboard load time.

### Model retraining cadence

| Trigger | Action |
|---------|--------|
| Weekly (Monday 03:00 UTC) | ADF pipeline runs inference only (no retraining) |
| Monthly or when val MAE drifts > 2 min | Re-run notebooks 12 → 13 → 14 → 15; promote new `lgbm_atd_model.pkl` to model registry |
| New territory / geo_archetype added | Re-run from notebook 11 to re-encode categoricals |